# Ube Craze Sentiment Visualizations

This notebook generates visualizations from the final sentiment-scored dataset to illustrate the dynamics of Filipino gastronationalism (positive pride) versus potential cultural dilution (negative/neutral/exoticizing themes).

## 1. Imports and Config

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.util import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from ube_craze_nlp.nlp.clean import tokenize_text, remove_stopwords
from ube_craze_nlp.utils.paths import FINAL_DATA_DIR, OUTPUTS_DIR, ensure_dirs

ensure_dirs()

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

print(f"Figures will be exported to: {OUTPUTS_DIR}")

## 2. Load Scored Dataset

In [ ]:
final_file = FINAL_DATA_DIR / "sentiment_scored_comments.csv"

if not final_file.exists():
    print("❌ Scored dataset not found! Please run Notebook 03 first.")
    df_comments = pd.DataFrame()
else:
    df_comments = pd.read_csv(final_file)
    print(f"Loaded {len(df_comments)} scored comments.")
    display(df_comments.head())

## 3. Visualize Sentiment Distribution

We plot the overall sentiment distribution (Positive, Neutral, Negative) to understand public reception of the Ube craze on TikTok.

In [ ]:
if not df_comments.empty:
    # Compute counts and percentages
    sentiment_counts = df_comments["sentiment"].value_counts()
    sentiment_pct = df_comments["sentiment"].value_counts(normalize=True) * 100

    # Plot
    palette = {"positive": "#6a0dad", "neutral": "#a295a7", "negative": "#d46a83"}  # Purple, Grey, Muted Pink
    ax = sns.barplot(
        x=sentiment_counts.index,
        y=sentiment_counts.values,
        hue=sentiment_counts.index,
        palette=palette,
        legend=False
    )

    plt.title("TikTok Comment Sentiment on the Global Ube Craze", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Sentiment Label", fontsize=12)
    plt.ylabel("Number of Comments", fontsize=12)

    # Add text labels on top of the bars
    for i, p in enumerate(ax.patches):
        height = p.get_height()
        percentage = sentiment_pct.iloc[i]
        ax.annotate(
            f"{int(height)} ({percentage:.1f}%)",
            (p.get_x() + p.get_width() / 2., height),
            ha="center", va="center",
            xytext=(0, 9), textcoords="offset points",
            fontsize=11, fontweight="semibold"
        )

    sns.despine()
    plt.tight_layout()

    # Save plot
    fig_path = OUTPUTS_DIR / "sentiment_distribution.png"
    plt.savefig(fig_path, dpi=300)
    print(f"✅ Exported plot to: {fig_path}")
    plt.show()

## 4. Word Frequency Distributions (Pride vs. Exoticization)

We extract high-frequency terms from positive comments (potential indicators of national pride and Gastronationalism) and compare them to terms from neutral or negative comments (potential indicators of commercialization, exoticization, or cultural dilution).

In [ ]:
def get_frequent_words(texts_series, top_n=15):
    all_tokens = []
    for text in texts_series:
        tokens = tokenize_text(str(text))
        cleaned_tokens = remove_stopwords(tokens)
        all_tokens.extend(cleaned_tokens)
    return Counter(all_tokens).most_common(top_n)

if not df_comments.empty:
    # Separate positive and neutral/negative comment texts
    pos_comments = df_comments[df_comments["sentiment"] == "positive"]["normalized_text"]
    neg_neu_comments = df_comments[df_comments["sentiment"].isin(["negative", "neutral"])] ["normalized_text"]

    # Compute word counts
    pos_words = get_frequent_words(pos_comments)
    neg_neu_words = get_frequent_words(neg_neu_comments)

    # --- Plot Positive Words ---
    df_pos_words = pd.DataFrame(pos_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_pos_words, x="count", y="word", hue="word", palette="Purples_r", legend=False)
    plt.title("Top Words in Positive Comments (Cultural Pride / Gastronationalism)", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    pos_fig_path = OUTPUTS_DIR / "word_frequency_pride.png"
    plt.savefig(pos_fig_path, dpi=300)
    print(f"✅ Exported positive words plot to: {pos_fig_path}")
    plt.show()

    # --- Plot Neutral/Negative Words ---
    df_neg_words = pd.DataFrame(neg_neu_words, columns=["word", "count"])
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_neg_words, x="count", y="word", hue="word", palette="Reds_r", legend=False)
    plt.title("Top Words in Neutral & Negative Comments (Commercialization / Exoticization / Context)", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Frequency", fontsize=11)
    plt.ylabel("Word", fontsize=11)
    plt.tight_layout()
    neg_fig_path = OUTPUTS_DIR / "word_frequency_exotic.png"
    plt.savefig(neg_fig_path, dpi=300)
    print(f"✅ Exported neutral/negative words plot to: {neg_fig_path}")
    plt.show()

## 5. N-Gram (Bigram & Trigram) Analysis

We extract and visualize multi-word phrases (bigrams and trigrams) to capture meaningful combinations of words (e.g., "Trader Joes", "purple yam") that single-word unigram analysis breaks apart.

In [ ]:
def get_frequent_ngrams(texts_series, n, top_n=15):
    all_ngrams = []
    for text in texts_series:
        tokens = tokenize_text(str(text))
        cleaned_tokens = remove_stopwords(tokens)
        ngram_list = list(ngrams(cleaned_tokens, n))
        ngram_strings = [" ".join(gram) for gram in ngram_list]
        all_ngrams.extend(ngram_strings)
    return Counter(all_ngrams).most_common(top_n)

if not df_comments.empty:
    pos_comments = df_comments[df_comments["sentiment"] == "positive"]["normalized_text"]
    neg_neu_comments = df_comments[df_comments["sentiment"].isin(["negative", "neutral"])] ["normalized_text"]

    # --- Bigrams ---
    pos_bigrams = get_frequent_ngrams(pos_comments, n=2)
    neg_neu_bigrams = get_frequent_ngrams(neg_neu_comments, n=2)

    df_pos_bi = pd.DataFrame(pos_bigrams, columns=["phrase", "count"])
    df_neg_bi = pd.DataFrame(neg_neu_bigrams, columns=["phrase", "count"])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=df_pos_bi, x="count", y="phrase", ax=axes[0], hue="phrase", palette="Purples_r", legend=False)
    axes[0].set_title("Top Bigrams in Positive Comments (Pride)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Frequency")
    axes[0].set_ylabel("")

    sns.barplot(data=df_neg_bi, x="count", y="phrase", ax=axes[1], hue="phrase", palette="Reds_r", legend=False)
    axes[1].set_title("Top Bigrams in Neutral/Negative Comments (Exotic/Context)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Frequency")
    axes[1].set_ylabel("")

    plt.tight_layout()
    bi_fig_path = OUTPUTS_DIR / "word_frequency_bigrams.png"
    plt.savefig(bi_fig_path, dpi=300)
    print(f"✅ Exported bigram plot to: {bi_fig_path}")
    plt.show()

    # --- Trigrams ---
    pos_trigrams = get_frequent_ngrams(pos_comments, n=3)
    neg_neu_trigrams = get_frequent_ngrams(neg_neu_comments, n=3)

    df_pos_tri = pd.DataFrame(pos_trigrams, columns=["phrase", "count"])
    df_neg_tri = pd.DataFrame(neg_neu_trigrams, columns=["phrase", "count"])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    sns.barplot(data=df_pos_tri, x="count", y="phrase", ax=axes[0], hue="phrase", palette="Purples_r", legend=False)
    axes[0].set_title("Top Trigrams in Positive Comments (Pride)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Frequency")
    axes[0].set_ylabel("")

    sns.barplot(data=df_neg_tri, x="count", y="phrase", ax=axes[1], hue="phrase", palette="Reds_r", legend=False)
    axes[1].set_title("Top Trigrams in Neutral/Negative Comments (Exotic/Context)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Frequency")
    axes[1].set_ylabel("")

    plt.tight_layout()
    tri_fig_path = OUTPUTS_DIR / "word_frequency_trigrams.png"
    plt.savefig(tri_fig_path, dpi=300)
    print(f"✅ Exported trigram plot to: {tri_fig_path}")
    plt.show()

## 6. Unsupervised Topic Clustering (K-Means)

We use TF-IDF vectorization and K-Means clustering to discover latent topics in the comments dataset automatically, then evaluate the sentiment breakdown across these different topics.

In [ ]:
if not df_comments.empty:
    texts = df_comments["normalized_text"].fillna("").tolist()

    def get_clean_string(text):
        tokens = tokenize_text(str(text))
        return " ".join(remove_stopwords(tokens))

    clean_texts = [get_clean_string(t) for t in texts]
    non_empty_indices = [i for i, t in enumerate(clean_texts) if t.strip() != ""]
    
    if len(non_empty_indices) < 4:
        print("⚠️ Too few comments with valid tokens to cluster. Skipping clustering.")
    else:
        df_cluster = df_comments.iloc[non_empty_indices].copy()
        texts_for_tfidf = [clean_texts[i] for i in non_empty_indices]

        vectorizer = TfidfVectorizer(max_features=800)
        tfidf_matrix = vectorizer.fit_transform(texts_for_tfidf)

        n_clusters = min(4, len(df_cluster))
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df_cluster["topic_cluster"] = kmeans.fit_predict(tfidf_matrix)

        print("Top terms per cluster:")
        vocab = vectorizer.get_feature_names_out()
        centroids = kmeans.cluster_centers_
        
        cluster_labels = {}
        for i in range(n_clusters):
            centroid = centroids[i]
            sorted_indices = centroid.argsort()[::-1]
            top_terms = [vocab[idx] for idx in sorted_indices[:8]]
            terms_str = ", ".join(top_terms)
            print(f"  Cluster {i}: {terms_str}")
            cluster_labels[i] = f"Topic {i} ({top_terms[0]}, {top_terms[1]})"

        df_cluster["topic_label"] = df_cluster["topic_cluster"].map(cluster_labels)

        # Plot Sentiment Distribution per Cluster
        pivot_df = df_cluster.groupby(["topic_label", "sentiment"]).size().unstack(fill_value=0)
        pivot_df = pivot_df.reindex(columns=["positive", "neutral", "negative"], fill_value=0)
        pivot_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

        ax = pivot_pct.plot(
            kind="barh",
            stacked=True,
            color={"positive": "#6a0dad", "neutral": "#a295a7", "negative": "#d46a83"},
            figsize=(12, 6)
        )
        plt.title("Sentiment Distribution Across Unsupervised Topic Clusters", fontsize=13, fontweight="bold", pad=15)
        plt.xlabel("Percentage (%)", fontsize=11)
        plt.ylabel("Topic Cluster (Top Terms)", fontsize=11)
        plt.legend(title="Sentiment", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()

        cluster_fig_path = OUTPUTS_DIR / "topic_clusters_sentiment.png"
        plt.savefig(cluster_fig_path, dpi=300)
        print(f"✅ Exported topic cluster plot to: {cluster_fig_path}")
        plt.show()

## 7. Reply-Thread Sentiment Interaction

We construct parent-reply sentiment pairs to build an interaction heatmap, helping us study the conversational dynamics of digital gastronationalism (e.g., how the community responds to negative/exoticizing comments versus positive expressions of pride).

In [ ]:
if not df_comments.empty:
    df_replies = df_comments[df_comments["is_reply"] == True].copy()
    df_parents = df_comments[df_comments["is_reply"] == False].copy()

    if df_replies.empty:
        print("⚠️ No replies found in the dataset. Skipping reply-thread sentiment interaction.")
    else:
        df_replies["reply_to_comment_id"] = df_replies["reply_to_comment_id"].astype(str)
        df_parents["comment_id"] = df_parents["comment_id"].astype(str)

        df_pairs = df_replies.merge(
            df_parents,
            left_on="reply_to_comment_id",
            right_on="comment_id",
            suffixes=("_reply", "_parent")
        )

        if df_pairs.empty:
            print("⚠️ No matching parent comments found for replies. Skipping interaction matrix.")
        else:
            print(f"Constructed {len(df_pairs)} parent-reply sentiment pairs.")

            categories = ["positive", "neutral", "negative"]
            transition_matrix = pd.crosstab(
                df_pairs["sentiment_parent"],
                df_pairs["sentiment_reply"],
                dropna=False
            )
            
            transition_matrix = transition_matrix.reindex(index=categories, columns=categories, fill_value=0)
            transition_matrix_pct = transition_matrix.div(transition_matrix.sum(axis=1), axis=0).fillna(0) * 100

            plt.figure(figsize=(8, 6))
            sns.heatmap(
                transition_matrix_pct,
                annot=transition_matrix.astype(str) + "\n(" + transition_matrix_pct.round(1).astype(str) + "%)",
                fmt="",
                cmap="Purples",
                cbar_kws={'label': 'Percentage of replies (%)'},
                annot_kws={"size": 11, "weight": "bold"}
            )
            
            plt.title("TikTok Comment Sentiment Interaction Matrix\n(Parent Sentiment vs. Reply Sentiment)", fontsize=13, fontweight="bold", pad=15)
            plt.xlabel("Direct Reply Sentiment", fontsize=11)
            plt.ylabel("Parent Comment Sentiment", fontsize=11)
            plt.tight_layout()

            heatmap_path = OUTPUTS_DIR / "reply_sentiment_heatmap.png"
            plt.savefig(heatmap_path, dpi=300)
            print(f"✅ Exported reply sentiment heatmap to: {heatmap_path}")
            plt.show()